# 3.2 Coordinate Systems And Surface-Relative Coordinates

Coordinate systems let properties, sources, and receivers be defined relative to model geometry. This example adds sinusoidal topography, defines a velocity profile in depth below that topographic surface, and places sources/receivers with the `surface.below()` and `surface.on()` helpers.

By the end, you should be able to place properties, sources, and receivers in surface-relative coordinates and inspect the exported coordinate-system payload.


## How To Read This Tutorial

This notebook explains why coordinate systems exist in the API. Real surveys and near-surface models are often easier to describe relative to a surface than relative to a flat global axis. FrequenSolve lets those surface-relative coordinates travel with properties, sources, and receivers.

The story is intentionally concrete: build topography, define a depth-below-surface property, place devices on and below that surface, inspect the exported coordinate-system payload, then run the result.

## Design Notes

Coordinate systems let properties, sources, receivers, and output requests use coordinate frames other than raw global Cartesian coordinates. The public API supports raw arrays for simple cases and coordinate-aware objects when units or systems need to travel with the values.

| Concept | API pattern | Why it matters |
| --- | --- | --- |
| Global coordinates | `[[x, z], ...]` or Pint quantities | Compact for simple flat models. |
| Coordinate-aware values | `{value, units, system}` through helper objects | Preserves units and coordinate-system names in exported inputs. |
| Surface-relative systems | `sim.add_surface_coordinate_system(...)` | Defines coordinates such as depth below a named surface. |
| Surface point helpers | `surface.on(...)`, `surface.below(...)` | Places sources and receivers on or below variable topography without manual interpolation. |

In this notebook the velocity profile is a function of `below`, not global `z`. That distinction is the point: the material trend follows the topographic surface instead of staying horizontal in global coordinates.


## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:

import numpy as np
import xarray as xr
import frequensolve as fs

u = fs.ureg


## Surface-Relative Model And Acquisition

The model uses a sinusoidal top surface and a velocity profile defined by distance below that surface. The profile coordinate is named `below`, and the surface-relative coordinate system declares that `below` follows the inherited vertical direction with positive values downward.

The acquisition is surface-relative too. `surface.on(...)` places receivers directly on the variable topography, while `surface.below(...)` places the source a fixed physical distance below that same surface. That is safer than manually interpolating topography into global coordinates because the exported source/receiver payload keeps the coordinate-system metadata attached to the values.


In [ ]:
project = fs.Project(
    name="project",
    path="./scratch/tutorials/coordinate_systems",
    log_level="INFO",
)
sim = project.new_simulation(
    name="surface_relative",
    physics="acoustic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

x = np.linspace(0.0, 1.0, 201)
topography = xr.DataArray(
    0.02 * np.sin(2.0 * np.pi * x),
    dims=["x"],
    coords={"x": x},
    attrs={"units": "km"},
)
topography.coords["x"].attrs["units"] = "km"

depth = np.linspace(0.0, 0.5, 101)
vp_depth = xr.DataArray(
    1.5 + 1.0 * depth / depth.max(),
    dims=["below"],
    coords={"below": depth},
    attrs={"units": "km/s"},
)
vp_depth.coords["below"].attrs["units"] = "km"

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=topography)
model.add_layer(
    name="near_surface",
    properties={
        "Vp": {"value": vp_depth, "coordinate_system": "top_relative"},
        "Rho": 1.8 * u.g / u.cm**3,
    },
)
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model

top_system = sim.add_surface_coordinate_system(
    name="top_relative",
    surface="top",
    axes=[fs.Axis("below", direction="z", positive="down")],
)
surface = sim.model_surface("top", name="top_points", normal="down")

acq = fs.Acquisition()
acq.add_source_group(kind="scalar", coords=surface.below([[0.5, 0.02]], units="km"))
node = fs.ReceiverNode(name="hydrophone")
node.add_component(name="p", field="pressure")
acq.add_receiver_group(
    name="surface",
    device=node,
    coords=surface.on(np.linspace(0.1, 0.9, 21), units="km"),
)
sim += acq
model.plot("vp", figsize=(7, 3), aspect="equal")


## Inspect Surface-Relative Export

Before submitting the job, inspect the exported acquisition and coordinate-system payload. This is a useful habit whenever a model uses non-global coordinates: the payload should show explicit units, a coordinate-system name, and the surface-relative axis used by the property or receiver definition.

The contract accepts raw arrays for compatibility, but new tutorials should prefer coordinate-aware values when a coordinate is not simply "these numbers in the simulation's default length units." It makes solver inputs easier to audit and avoids silent unit assumptions.


In [ ]:
acq_payload = sim.acquisition.to_fs(sim.export_context())
{
    "source_coordinates": acq_payload["source_groups"][0]["source"]["coordinates"],
    "receiver_coordinate_type": acq_payload["receiver_groups"][0]["coordinates"].get("_type"),
    "coordinate_systems": [system.to_fs() for system in sim.coordinate_systems],
}


## Run With Surface-Relative Properties And Receivers

The mesh still lives in global coordinates. The surface-relative system is used only where requested: the `Vp` property and the acquisition coordinates.


In [ ]:
sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
job = fs.TimeDomainJob(
    name="time_surface_relative",
    simulation=sim,
    f_min=0.0,
    f_max=30.0,
    T_max=0.9,
)
result = site.submit(job).wait()
traces = result.traces(upscale=4)
traces.summary


## Plot Topographic Survey Traces

The trace gather is an end-to-end check of the surface-relative setup: topography was exported, the `below` property profile was interpreted relative to that surface, and the acquisition geometry followed the surface rather than a flat datum.

When debugging real topographic models, compare this gather with the exported acquisition payload above. If receiver coordinates look flat or the property trend looks horizontal in global depth, the problem is usually a missing coordinate-system name or coordinate unit metadata.


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
group = traces.groups[0]
component = traces.components(group)[0]
source = traces.sources(group)[0]
gather = traces.td(group, component, source, wavelet, upscale=4, T_max=0.9)
fs.plot_gather(
    gather,
    A=2.0 * np.nanstd(np.real(gather.values)),
    cmap="gray",
    figsize=(9, 4),
    title="Surface-relative acquisition response",
)


## Before Moving On

A coordinate-aware model should be reviewed twice: visually in model space and structurally in the exported payload. The payload should make units and coordinate-system names explicit. If a receiver or property looks shifted, inspect the coordinate system before changing the physics.

The useful mental model is that the mesh still lives globally, while selected inputs can be authored in a more natural local frame.

## Result Review Checklist

After the run succeeds, check three things before trusting a surface-relative setup:

| Check | What good output looks like | Common mistake |
| --- | --- | --- |
| Acquisition payload | Source and receiver coordinates carry the intended surface-relative system and units. | Coordinates silently treated as global Cartesian values. |
| Model plot/sample | The velocity trend follows depth below topography, not absolute `z`. | Missing coordinate-system metadata on the property array. |
| Trace gather | Receiver moveout is consistent with the topographic line rather than a flat datum. | Receivers were manually flattened or units were stripped before export. |

For production topographic models, save the acquisition payload next to the run notes. It is usually the fastest artifact to inspect when a receiver line appears shifted or flattened in downstream processing.
